# PTB-XL exploration

Goal: sanity-check the SCP → 5-class mapping against `scp_statements.csv`, look at code prevalence per fold, and eyeball a few example waveforms per class.

Outputs of this notebook gate the final label list in `smartecg/data/labels.py`.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

from smartecg.data.labels import build_label_table, CLASSES, CLASS_CODE_MAP

ROOT = Path(os.environ.get('PTBXL_DATA_DIR', 'data/raw/ptbxl'))
scp = pd.read_csv(ROOT / 'scp_statements.csv', index_col=0)
scp.head()

,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


In [2]:
# every code we map should exist in scp_statements
all_mapped = set().union(*CLASS_CODE_MAP.values())
missing = all_mapped - set(scp.index)
assert not missing, f'codes in mapping not present in scp_statements.csv: {missing}'

In [3]:
df = build_label_table(ROOT / 'ptbxl_database.csv')
print(df.shape)
df[[f'y_{c}' for c in CLASSES]].sum()

(21799, 13)


y_normal        9438.0
y_af            1570.0
y_stemi         4134.0
y_arrhythmia    3951.0
y_conduction    4891.0
dtype: float32

In [4]:
# class prevalence per fold — make sure no fold is starved of a class
per_fold = df.groupby('strat_fold')[[f'y_{c}' for c in CLASSES]].mean()
per_fold

,y_normal,y_af,y_stemi,y_arrhythmia,y_conduction
strat_fold,,,,,
1,0.429425,0.072184,0.192184,0.184368,0.220690
2,0.441541,0.071527,0.184319,0.177900,0.221458
3,0.447536,0.071624,0.185675,0.180201,0.222172
4,0.423643,0.072217,0.190892,0.181693,0.227231
5,0.428243,0.072677,0.195952,0.182153,0.228151
6,0.423838,0.071330,0.194202,0.178095,0.226875
7,0.443934,0.072610,0.186581,0.183364,0.219210
8,0.425679,0.073171,0.189139,0.181776,0.225495
9,0.431516,0.071461,0.188731,0.181402,0.226752
